# PAL-Net Ortodontik Landmark Lokalizasyonu - Colab GPU

Bu notebook PAL-Net modelini 23 noktali 3B ortodontik dataset uzerinde Google Colab GPU ile calistirir. PAL-Net, DiffusionNet ve PointNet++ karsilastirmasi icin ayni ortak split dosyasi kullanilir.

## 1. Drive bagla ve kodu hazirla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!test -d comparative-study || git clone https://github.com/eckdev/comparative-study.git
%cd /content/comparative-study
!git pull
!python -m pip install -q -r palnet_orthodontic_comparison/requirements.txt

## 2. Veri yollarini ayarla

In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/orthodontic/data/dataset')
SPLITS_JSON = Path('/content/comparative-study/shared_splits/orthodontic_180_60_60_seed42.json')
TRANSFORM_DIR = Path('/content/drive/MyDrive/orthodontic/transforms/orthodontic_procrustes_rigid_20260627_143801')
RUN_ROOT = Path('/content/drive/MyDrive/orthodontic/palnet_runs')
USE_TRANSFORMS = TRANSFORM_DIR.exists()
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT exists:', DATA_ROOT.exists(), DATA_ROOT)
print('SPLITS_JSON exists:', SPLITS_JSON.exists(), SPLITS_JSON)
print('TRANSFORM_DIR exists:', TRANSFORM_DIR.exists(), TRANSFORM_DIR)
print('RUN_ROOT:', RUN_ROOT)

## 3. GPU ve dataset kontrolu

In [ ]:
import glob
import json
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

ply_count = len(glob.glob(str(DATA_ROOT / 'Class*' / '*' / '*.ply')))
txt_count = len(glob.glob(str(DATA_ROOT / 'Class*' / 'Class*-Landmark' / '*' / '*.txt')))
print('PLY:', ply_count)
print('Landmark TXT:', txt_count)

split_payload = json.loads(SPLITS_JSON.read_text())
print('Split train/val/test:', split_payload['n_train'], split_payload['n_val'], split_payload['n_test'])

assert DATA_ROOT.exists(), 'DATA_ROOT bulunamadi. Google Drive yolunu kontrol et.'
assert SPLITS_JSON.exists(), 'Ortak split dosyasi bulunamadi. Repoyu guncelle.'
assert ply_count > 0 and txt_count > 0, 'Dataset eksik gorunuyor.'

## 4. Smoke test

In [ ]:
import shlex
import os
import subprocess

%cd /content/comparative-study/palnet_orthodontic_comparison/upstream

smoke_dir = RUN_ROOT / 'palnet_smoke_fast_colab'
cmd = [
    'python', '-u', 'run_orthodontic.py',
    '--data-root', str(DATA_ROOT),
    '--splits-json', str(SPLITS_JSON),
    '--output-dir', str(smoke_dir),
    '--epochs', '1',
    '--patience', '1',
    '--batch-size', '8',
    '--patch-size', '100',
    '--surface-points', '1024',
    '--snap-k', '1',
    '--max-train-samples', '12',
    '--max-val-samples', '6',
    '--max-test-samples', '6',
]
if USE_TRANSFORMS:
    cmd.extend(['--transformation-dir', str(TRANSFORM_DIR)])
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['TQDM_MININTERVAL'] = '1'
print(' '.join(shlex.quote(x) for x in cmd), flush=True)
subprocess.run(cmd, check=True, env=env)

## 5. Colab Pro pratik kosu

In [ ]:
main_dir = RUN_ROOT / 'palnet_procrustes_p500_surface50k_e120'
cmd = [
    'python', '-u', 'run_orthodontic.py',
    '--data-root', str(DATA_ROOT),
    '--splits-json', str(SPLITS_JSON),
    '--output-dir', str(main_dir),
    '--epochs', '120',
    '--patience', '30',
    '--batch-size', '4',
    '--patch-size', '500',
    '--surface-points', '50000',
    '--lr', '0.001',
    '--snap-k', '1',
    '--model', 'PALNET',
    '--loss', 'combined',
]
if USE_TRANSFORMS:
    cmd.extend(['--transformation-dir', str(TRANSFORM_DIR)])
print(' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)

## 6. A100 / yuksek VRAM buyuk kosu

In [ ]:
big_dir = RUN_ROOT / 'palnet_refiner_p1000_surface100k_e200'
big_cmd = [
    'python', '-u', 'run_orthodontic.py',
    '--data-root', str(DATA_ROOT),
    '--splits-json', str(SPLITS_JSON),
    '--output-dir', str(big_dir),
    '--epochs', '200',
    '--patience', '40',
    '--batch-size', '2',
    '--patch-size', '1000',
    '--surface-points', '100000',
    '--lr', '0.001',
    '--snap-k', '1',
    '--model', 'PALNET',
    '--loss', 'combined',
    '--template-mode', 'class_gender',
    '--train-refiner',
    '--refine-center', 'stage1',
    '--residual-target',
    '--landmark-weighting', 'val_error',
    '--center-jitter-mm', '2.0',
    '--point-noise-mm', '0.1',
    '--point-dropout', '0.05',
    '--refiner-snap-k-candidates', '1,3,5',
]
if USE_TRANSFORMS:
    big_cmd.extend(['--transformation-dir', str(TRANSFORM_DIR)])
print('A100 / yuksek VRAM buyuk kosu basliyor:')
print(' '.join(shlex.quote(x) for x in big_cmd))
subprocess.run(big_cmd, check=True)

## 7. Sonuclari oku

In [ ]:
import json

for run_dir in sorted(RUN_ROOT.glob('palnet_*')):
    metrics_path = run_dir / 'metrics.json'
    if not metrics_path.exists():
        continue
    metrics = json.loads(metrics_path.read_text())
    snapped = metrics.get('palnet_snapped', {})
    raw = metrics.get('palnet_raw', {})
    print(run_dir.name)
    print('  snapped ALE:', snapped.get('ale'), 'median:', snapped.get('median'))
    print('  raw ALE:', raw.get('ale'), 'median:', raw.get('median'))
    refined_path = run_dir / 'metrics_refined.json'
    if refined_path.exists():
        refined = json.loads(refined_path.read_text()).get('palnet_refined_snapped', {})
        print('  refined ALE:', refined.get('ale'), 'median:', refined.get('median'))